# Durable-execution security lab

Scenario: resume a paused support workflow safely. The default path uses deterministic, credential-free controls and exposes policy receipts—not model reasoning.

## Objectives and invariants

An unsafe resume, duplicate side effect, or write after revocation must be rejected. A valid, unchanged, reauthorized resume may proceed.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[1] / 'shared'))
from runtime_security_lab import Run, release_gate


## Baseline: idempotent action and revocation

The runtime records action IDs and refuses both a duplicate effect and a new write after the capability is revoked.

In [ ]:
run = Run('notebook-run', 'north', 'state-v1', 'policy-v1')
assert run.commit_once('close-case-7')
assert not run.commit_once('close-case-7')
run.revoke('writes')
assert not run.commit_once('close-case-8')
run.trace


## Failure injection and retest

A changed policy version forces a pause. The release gate independently blocks severe failures even when ordinary evidence exists.

In [ ]:
assert not run.resume(policy_version='policy-v2', checkpoint_version='state-v1', authorized=True)
assert not release_gate({'threat_model', 'attack_suite', 'trace_coverage', 'rollback_drill', 'owner'}, 1)['ready']
print('Unsafe resume and severe release failure were blocked.')


## Production upgrade

Add checkpoint integrity, approval expiry, external tool receipts, event deduplication, version migration rules, and a recovery drill. Exercise: model a tool-version change and state whether it should require reapproval.